# EXP-01 — Treino segmentação (CBIS-DDSM)
## Treino segmentação unet vs resnet34

Notebook equivalente ao comando local/Docker (`train.py --config config_exp01.yaml`
+ `predict.py`), com rastreamento de experimentos no MLflow seguindo o
**Padrão de tracking — MLflow** (tags, parâmetros, métricas e artefatos
obrigatórios).

**Dependências:** S3-01, S3-02 e configuração mínima do MLflow (servidor já
provisionado — este notebook só se conecta a ele via `MLFLOW_TRACKING_URI`,
não configura o servidor).

**Fluxo deste notebook:**
1. Checklist de entrada (dataset/manifest, split, seed, config, ambiente)
2. Conexão ao MLflow (credenciais via Kaggle Secrets, nunca hardcoded)
3. **Smoke test** — obrigatório rodar e passar antes do run completo
4. Run completo — treino + predição no teste congelado + métricas com IC 95%

**Segurança de dados (crítico):** nenhum arquivo DICOM, imagem clínica,
caminho pessoal de máquina ou credencial é publicado neste notebook ou no
seu output. Checkpoints e CSVs de predição vão *apenas* para o MLflow
(artifact store próprio, não o output público do Kaggle) e são apagados da
pasta de trabalho imediatamente após o upload.

## Checklist de entrada (leitura manual antes de rodar)

Confirme cada item **antes** de rodar a célula do smoke test. A célula de
código logo abaixo também valida boa parte disso automaticamente, mas a
leitura manual continua sendo a checagem principal.

| Item | Confirmação |
|---|---|
| **Dataset / manifest** | O dataset CBIS-DDSM está anexado ao notebook (`/kaggle/input/...`) e os 4 CSVs oficiais (mass/calc × train/test) existem no caminho configurado |
| **Split** | `train_csvs` e `test_csvs` apontam para arquivos diferentes — o teste é o split oficial do CBIS-DDSM, nunca re-sorteado |
| **Seed** | `CONFIG["seed"]` está definida e é a mesma usada em runs anteriores comparáveis |
| **Configuração** | `CONFIG` revisado: arquitetura, resolução, batch, LR, épocas correspondem ao experimento pretendido (ex: EXP-01 na matriz de experimentos) |
| **Ambiente** | Rodando em Kaggle (ou nuvem aprovada) com GPU do tipo esperado — confira em Settings > Accelerator |
| **Segredos MLflow** | `MLFLOW_TRACKING_URI` (e token, se aplicável) cadastrados em Add-ons > Secrets — nunca colados no código |
| **Orçamento de GPU** | Custo estimado deste run já checado contra o limite semanal (ver planilha de plano de execução) antes de rodar o run completo |


In [1]:
import os

base = "/kaggle/input/datasets"
for root, dirs, files in os.walk(base):
    depth = root.replace(base, '').count(os.sep)
    if depth > 3:
        dirs[:] = []
        continue
    print(root)
    for f in files[:5]:
        print("   ", f)

/kaggle/input/datasets
/kaggle/input/datasets/awsaf49
/kaggle/input/datasets/awsaf49/cbis-ddsm-breast-cancer-image-dataset
/kaggle/input/datasets/awsaf49/cbis-ddsm-breast-cancer-image-dataset/jpeg
/kaggle/input/datasets/awsaf49/cbis-ddsm-breast-cancer-image-dataset/csv
    dicom_info.csv
    mass_case_description_train_set.csv
    calc_case_description_train_set.csv
    meta.csv
    calc_case_description_test_set.csv
/kaggle/input/datasets/joovitorbach
/kaggle/input/datasets/joovitorbach/mlflowdb
    mlflow.db
/kaggle/input/datasets/joovitorbach/evaluate-py-compartilhado
/kaggle/input/datasets/joovitorbach/evaluate-py-compartilhado/evaluate_shared


In [2]:
import os
os.environ['CODE_VERSION'] = 'eac0e08'


import os, sys, subprocess, json, hashlib, time
from datetime import datetime

# ============================================================
# CONFIG — edite apenas estes valores antes de rodar.
# NUNCA coloque caminho pessoal de máquina local aqui (ex: C:\Users\...).
# Neste notebook, os dados vêm sempre de /kaggle/input/... (dataset Kaggle).
# ============================================================
CONFIG = {
    "experiment_id": "EXP-01",
    "owner": "Guilherme BettegaS",          # <-- troque pelo seu usuário/identificador de papel
    "role": "02",                     # <-- número do papel responsável (ver padrão de tags)
    "ablation_axis": "resolution",    # eixo variado nesta rodada (baseline = ponto de partida do eixo "resolution")
    "seed": 42,
    "architecture": "resnet50",
    "pretrained_weights": "imagenet",
    "image_size": 224,
    "batch_size": 32,
    "learning_rate": 1e-4,
    "weight_decay": 1e-5,
    "epochs": 20,
    "dataset_root": "/kaggle/input/datasets/awsaf49/cbis-ddsm-breast-cancer-image-dataset",
    "train_csvs": [
        "csv/mass_case_description_train_set.csv",
        "csv/calc_case_description_train_set.csv",
    ],
    "test_csvs": [
        "csv/mass_case_description_test_set.csv",
        "csv/calc_case_description_test_set.csv",
    ],
    "image_path_column": "cropped image file path",
    "patient_id_column": "patient_id",
    "pathology_column": "pathology",
    "malignant_labels": ["MALIGNANT"],
    "non_malignant_labels": ["BENIGN", "BENIGN_WITHOUT_CALLBACK"],
    "val_fraction": 0.15,
    "preprocessing_version": "preprocessing@REPLACE_WITH_COMMIT_OR_TAG",
}

WORKDIR = "/kaggle/working"
os.makedirs(WORKDIR, exist_ok=True)


def get_code_version():
    """Hash do commit Git ativo. Em notebooks Kaggle não há .git local por
    padrão — se este notebook foi gerado a partir de um script versionado,
    defina a variável de ambiente CODE_VERSION antes de rodar (ex: via
    Kaggle Secrets ou como parâmetro do kernel). Caso contrário, cai para
    um placeholder explícito (NUNCA deixe isso como 'unknown' num run real)."""
    env_version = os.environ.get("CODE_VERSION")
    if env_version:
        return env_version
    try:
        out = subprocess.check_output(["git", "rev-parse", "HEAD"], stderr=subprocess.DEVNULL)
        return out.decode().strip()
    except Exception:
        return "UNKNOWN_SET_CODE_VERSION_ENV_VAR"


def detect_environment():
    """Detecta o ambiente de execução para a tag `environment`."""
    if os.path.exists("/kaggle"):
        try:
            gpu_info = subprocess.check_output(
                ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                stderr=subprocess.DEVNULL,
            ).decode().strip()
            return "kaggle-gpu" if gpu_info else "kaggle-cpu"
        except Exception:
            return "kaggle-cpu"
    return "local-cpu"


CODE_VERSION = get_code_version()
ENVIRONMENT = detect_environment()

print(f"CODE_VERSION: {CODE_VERSION}")
print(f"ENVIRONMENT detectado: {ENVIRONMENT}")


CODE_VERSION: eac0e08
ENVIRONMENT detectado: kaggle-gpu


### Checklist de entrada — validação automática

In [3]:
# ============================================================
# CHECKLIST DE ENTRADA — roda automaticamente, mas NÃO substitui a leitura
# manual da checklist.md antes de aprovar a execução completa.
# Falha (assert) se algo obrigatório estiver faltando, em vez de deixar o
# treino começar com uma configuração incompleta.
# ============================================================

checklist_status = {}

# 1. Dataset / manifest presente
dataset_ok = os.path.isdir(CONFIG["dataset_root"])
checklist_status["dataset_root_existe"] = dataset_ok

manifest_files_ok = []
for csv_list_name in ["train_csvs", "test_csvs"]:
    for rel_path in CONFIG[csv_list_name]:
        full_path = os.path.join(CONFIG["dataset_root"], rel_path)
        manifest_files_ok.append(os.path.exists(full_path))
checklist_status["todos_os_csvs_encontrados"] = all(manifest_files_ok)

# 2. Split — confirma que vamos usar o split OFICIAL (arquivos de teste
# separados dos de treino, nunca a mesma lista)
checklist_status["split_train_test_distintos"] = (
    set(CONFIG["train_csvs"]).isdisjoint(set(CONFIG["test_csvs"]))
)

# 3. Seed definida
checklist_status["seed_definida"] = isinstance(CONFIG["seed"], int)

# 4. Configuração sem caminho pessoal de máquina local
config_str = json.dumps(CONFIG)
personal_path_patterns = ["C:\\Users", "/home/", "/Users/"]
checklist_status["sem_caminho_pessoal_no_config"] = not any(
    p in config_str for p in personal_path_patterns
)

# 5. Ambiente detectado é um ambiente aprovado (Kaggle ou nuvem aprovada)
checklist_status["ambiente_aprovado"] = ENVIRONMENT.startswith("kaggle") or ENVIRONMENT == "docker"

# 6. code_version não é o placeholder de erro
checklist_status["code_version_definida"] = CODE_VERSION != "UNKNOWN_SET_CODE_VERSION_ENV_VAR"

print("=== CHECKLIST DE ENTRADA ===")
for item, ok in checklist_status.items():
    print(f"  [{'OK' if ok else 'FALTA'}] {item}")

obrigatorios = ["dataset_root_existe", "todos_os_csvs_encontrados",
                 "split_train_test_distintos", "seed_definida",
                 "sem_caminho_pessoal_no_config", "ambiente_aprovado"]
faltando = [k for k in obrigatorios if not checklist_status[k]]

if faltando:
    raise RuntimeError(
        f"Checklist de entrada incompleto, itens faltando: {faltando}. "
        "Corrija antes de rodar o smoke test ou o treino completo."
    )

if not checklist_status["code_version_definida"]:
    print("\n[AVISO] CODE_VERSION não definida via variável de ambiente. "
          "Defina antes do run completo (não bloqueante para o smoke test).")

print("\nChecklist obrigatório: OK. Pode prosseguir para o smoke test.")


=== CHECKLIST DE ENTRADA ===
  [OK] dataset_root_existe
  [OK] todos_os_csvs_encontrados
  [OK] split_train_test_distintos
  [OK] seed_definida
  [OK] sem_caminho_pessoal_no_config
  [OK] ambiente_aprovado
  [OK] code_version_definida

Checklist obrigatório: OK. Pode prosseguir para o smoke test.


### Dataset — carregamento dos CSVs oficiais e split por paciente

In [5]:
# dataset.py
"""Dataset do CBIS-DDSM para classificação binária maligno x não maligno em
patches (imagens já recortadas na ROI da lesão).

CORREÇÃO IMPORTANTE (descoberta ao rodar no Kaggle): a coluna
'cropped image file path' dos CSVs mass/calc_case_description_*.csv aponta
para o caminho do arquivo DICOM ORIGINAL (.dcm), que não existe mais nesse
download — só as versões .jpg convertidas existem, dentro de jpeg/<SeriesInstanceUID>/.
Para achar o .jpg certo, usamos o csv/dicom_info.csv como tabela de tradução:
ele tem uma linha por imagem, com 'SeriesInstanceUID' e 'image_path' (o
caminho relativo real do .jpg, tipo 'CBIS-DDSM/jpeg/<UID>/1-172.jpg').

A tradução funciona assim:
    1. Extrai o SeriesInstanceUID do caminho bruto (.dcm) do CSV de casos
       — é o penúltimo segmento do caminho, logo antes do nome do arquivo.
    2. Procura esse SeriesInstanceUID no dicom_info.csv.
    3. Usa o 'image_path' correspondente (já aponta pro .jpg real).
"""

import os
import re
from typing import List, Optional

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms


# Um UID de DICOM é uma sequência de números separados por ponto (ex:
# "1.3.6.1.4.1.9590.100.1.2.128573...").
_UID_PATTERN = re.compile(r"^\d+(\.\d+)+$")


def load_and_merge_csvs(dataset_root: str, csv_paths: List[str]) -> pd.DataFrame:
    dfs = []
    for rel_path in csv_paths:
        full_path = os.path.join(dataset_root, rel_path)
        if not os.path.exists(full_path):
            raise FileNotFoundError(
                f"CSV não encontrado: {full_path}\n"
                "Confira 'dataset_root' e 'train_csvs'/'test_csvs' no config."
            )
        df = pd.read_csv(full_path)
        df.columns = [c.strip() for c in df.columns]
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)


def label_from_pathology(pathology: str, malignant_labels, non_malignant_labels) -> Optional[int]:
    pathology = str(pathology).strip().upper()
    if pathology in [p.upper() for p in malignant_labels]:
        return 1
    if pathology in [p.upper() for p in non_malignant_labels]:
        return 0
    return None


def extract_series_uid(raw_path: str) -> Optional[str]:
    """Extrai o SeriesInstanceUID de um caminho bruto do CSV de casos.

    Exemplo de entrada:
      '.../Mass-Training_P_00004_RIGHT_MLO_1/1.3.6.1...774/1.3.6.1...846/000000.dcm'
    O SeriesInstanceUID é o segmento de UID imediatamente ANTES do nome do
    arquivo (o mais profundo dos dois UIDs no caminho).
    """
    raw_path = str(raw_path).replace("\\", "/").strip()
    parts = [p for p in raw_path.split("/") if p]
    if len(parts) < 2:
        return None
    # O penúltimo segmento (logo antes do nome do arquivo .dcm) é o SeriesInstanceUID
    candidate = parts[-2]
    if _UID_PATTERN.match(candidate):
        return candidate
    # fallback: procura o último segmento que pareça um UID, de trás pra frente
    for p in reversed(parts[:-1]):
        if _UID_PATTERN.match(p):
            return p
    return None


def build_dicom_info_lookup(dataset_root: str, dicom_info_rel_path: str = "csv/dicom_info.csv") -> dict:
    """SeriesInstanceUID -> {"crop": ..., "mask": ..., "full": ..., "single": ...}
    (caminhos relativos a dataset_root). Em séries com 2 arquivos, o papel é
    decidido pela SeriesDescription quando as DUAS linhas a trazem; caso
    contrário, pelo tamanho: a máscara tem o tamanho da mamografia inteira e
    o recorte é sempre menor."""
    full_path = os.path.join(dataset_root, dicom_info_rel_path)
    if not os.path.exists(full_path):
        raise FileNotFoundError(f"dicom_info.csv não encontrado em {full_path}.")
    df = pd.read_csv(full_path)
    df.columns = [c.strip() for c in df.columns]
 
    for col in ["SeriesInstanceUID", "image_path", "SeriesDescription", "Rows", "Columns"]:
        if col not in df.columns:
            raise KeyError(f"dicom_info.csv sem a coluna '{col}'. Colunas: {list(df.columns)}")
 
    df["_uid"] = df["SeriesInstanceUID"].astype(str).str.strip()
    df["_path"] = df["image_path"].astype(str).str.strip().str.replace("\\", "/", regex=False)
    df["_path"] = df["_path"].apply(lambda p: p[len("CBIS-DDSM/"):] if p.startswith("CBIS-DDSM/") else p)
    df["_area"] = pd.to_numeric(df["Rows"], errors="coerce") * pd.to_numeric(df["Columns"], errors="coerce")
    desc_to_role = {"cropped images": "crop", "ROI mask images": "mask", "full mammogram images": "full"}
    df["_role"] = df["SeriesDescription"].astype(str).str.strip().map(desc_to_role)
 
    lookup, n_pairs, n_conflicts = {}, 0, 0
    for uid, g in df.groupby("_uid", sort=False):
        entry = {}
        if len(g) == 1:
            r = g.iloc[0]
            entry["single"] = r["_path"]
            if isinstance(r["_role"], str):
                entry[r["_role"]] = r["_path"]
        else:
            n_pairs += 1
            g = g.sort_values("_area", na_position="last")
            by_area = {"crop": g.iloc[0], "mask": g.iloc[-1]}  # menor = recorte, maior = máscara
            if set(g["_role"]) == {"crop", "mask"}:            # descrição completa nas duas linhas
                chosen = {r["_role"]: r for _, r in g.iterrows()}
                if chosen["crop"]["_path"] != by_area["crop"]["_path"]:
                    n_conflicts += 1
            else:
                chosen = by_area
            entry["crop"], entry["mask"] = chosen["crop"]["_path"], chosen["mask"]["_path"]
        lookup[uid] = entry
 
    print(f"[lookup] {len(lookup)} séries; {n_pairs} com par recorte/máscara; "
          f"{n_conflicts} em que a descrição contradiz o tamanho.")
    return lookup


def resolve_image_path(dataset_root: str, raw_path: str, dicom_lookup: dict, role: str = "crop") -> str:
    """Traduz o caminho bruto do CSV (.dcm) para o .jpg real. role: "crop"
    (padrão, usado pelo classificador), "mask" (máscara ROI) ou "full"
    (mamografia inteira)."""
    series_uid = extract_series_uid(raw_path)
 
    if series_uid and series_uid in dicom_lookup:
        entry = dicom_lookup[series_uid]
        rel = entry.get(role) or entry.get("single")
        if rel:
            return os.path.join(dataset_root, rel)
 
    raw_path_clean = str(raw_path).replace("\\", "/").strip()
    if os.path.isabs(raw_path_clean) and os.path.exists(raw_path_clean):
        return raw_path_clean
 
    marker = "jpeg/"
    idx = raw_path_clean.find(marker)
    if idx != -1:
        candidate = os.path.join(dataset_root, raw_path_clean[idx:])
        if os.path.exists(candidate):
            return candidate
 
    return os.path.join(dataset_root, raw_path_clean.lstrip("/"))


class CBISDDSMPatchDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, dataset_root: str, image_path_column: str,
                 resolution: int, normalize_mean, normalize_std, augment: bool = False,
                 dicom_lookup: Optional[dict] = None):
        self.df = dataframe.reset_index(drop=True)
        self.dataset_root = dataset_root
        self.image_path_column = image_path_column
        self.dicom_lookup = dicom_lookup or {}

        if augment:
            self.transform = transforms.Compose([
                transforms.Resize((resolution, resolution)),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomVerticalFlip(p=0.2),
                transforms.RandomRotation(degrees=10),
                transforms.ToTensor(),
                transforms.Normalize(mean=normalize_mean, std=normalize_std),
            ])
        else:
            self.transform = transforms.Compose([
                transforms.Resize((resolution, resolution)),
                transforms.ToTensor(),
                transforms.Normalize(mean=normalize_mean, std=normalize_std),
            ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = resolve_image_path(self.dataset_root, row[self.image_path_column], self.dicom_lookup)

        try:
            img = Image.open(img_path).convert("RGB")
        except Exception as e:
            raise RuntimeError(
                f"Falha ao abrir imagem '{img_path}' (linha {idx} do dataframe, "
                f"paciente {row.get('patient_id', '?')}). Caminho original no CSV: "
                f"'{row[self.image_path_column]}'. Erro original: {e}"
            )

        img_t = self.transform(img)
        label = torch.tensor(float(row["label"]), dtype=torch.float32)

        return img_t, label, str(row.get("patient_id", "")), img_path


def build_labeled_dataframe(dataset_root: str, csv_paths: List[str], malignant_labels,
                             non_malignant_labels, patient_id_column: str,
                             pathology_column: str) -> pd.DataFrame:
    df = load_and_merge_csvs(dataset_root, csv_paths)

    if patient_id_column not in df.columns:
        raise KeyError(
            f"Coluna '{patient_id_column}' não encontrada no CSV. "
            f"Colunas disponíveis: {list(df.columns)}"
        )
    if pathology_column not in df.columns:
        raise KeyError(
            f"Coluna '{pathology_column}' não encontrada no CSV. "
            f"Colunas disponíveis: {list(df.columns)}"
        )

    df["label"] = df[pathology_column].apply(
        lambda p: label_from_pathology(p, malignant_labels, non_malignant_labels)
    )
    n_before = len(df)
    df = df.dropna(subset=["label"]).reset_index(drop=True)
    n_after = len(df)
    if n_after < n_before:
        print(f"[dataset] Descartadas {n_before - n_after} linhas com rótulo de patologia "
              f"desconhecido (fora de {malignant_labels + non_malignant_labels}).")

    df["patient_id"] = df[patient_id_column]
    return df


def filter_resolvable_rows(df: pd.DataFrame, dataset_root: str, image_path_column: str,
                            dicom_lookup: dict) -> pd.DataFrame:
    """Remove linhas cujo arquivo de imagem não existe de fato no disco,
    APÓS a tradução via dicom_info.csv. Isso evita que o treino inteiro
    trave por causa de algumas linhas com dados inconsistentes no CSV
    original (comum em datasets grandes reais) — em vez disso, avisa
    quantas/quais foram descartadas e segue com as válidas."""
    resolved_paths = []
    exists_flags = []

    for raw_path in df[image_path_column]:
        resolved = resolve_image_path(dataset_root, raw_path, dicom_lookup)
        resolved_paths.append(resolved)
        exists_flags.append(os.path.exists(resolved))

    df = df.copy()
    df["_resolved_path"] = resolved_paths
    df["_path_exists"] = exists_flags

    n_before = len(df)
    missing_df = df[~df["_path_exists"]]
    df = df[df["_path_exists"]].reset_index(drop=True)
    n_after = len(df)

    if n_after < n_before:
        n_missing = n_before - n_after
        pct = 100 * n_missing / n_before
        print(f"[dataset] AVISO: {n_missing} de {n_before} linhas ({pct:.1f}%) descartadas "
              f"por não terem arquivo de imagem correspondente no disco "
              f"(inconsistência entre o CSV de casos e o dicom_info.csv/pasta jpeg/).")
        if n_missing <= 10:
            print("[dataset] Pacientes afetados:", missing_df["patient_id"].tolist())
        if pct > 5:
            print(f"[dataset] ATENÇÃO: {pct:.1f}% é uma proporção alta — vale investigar "
                  f"antes de confiar nos resultados (pode ser um problema sistemático, "
                  f"não só ruído aleatório).")

    return df.drop(columns=["_resolved_path", "_path_exists"])


def patient_level_split(df: pd.DataFrame, val_fraction: float, seed: int):
    rng = np.random.RandomState(seed)
    patients = df["patient_id"].unique()
    rng.shuffle(patients)

    n_val = max(1, int(len(patients) * val_fraction))
    val_patients = set(patients[:n_val])

    val_df = df[df["patient_id"].isin(val_patients)].reset_index(drop=True)
    train_df = df[~df["patient_id"].isin(val_patients)].reset_index(drop=True)

    return train_df, val_df


In [6]:
# ------------------------------------------------------------------------
# VALIDAÇÃO -- rode DEPOIS de aplicar o patch e reiniciar o kernel. Se o
# patch funcionou: contaminação ~0 no recorte, máscara maior que o recorte,
# e máscara com o mesmo tamanho da mamografia inteira.
# ------------------------------------------------------------------------
def validar_resolucao(config, dicom_lookup, n_amostra=300):
    from PIL import Image
    root = config["dataset_root"]
    img_col, mask_col, full_col = config["image_path_column"], "ROI mask file path", "image file path"
    for nome, csvs in [("treino", config["train_csvs"]), ("teste", config["test_csvs"])]:
        d = build_labeled_dataframe(root, csvs, config["malignant_labels"], config["non_malignant_labels"],
                                     config["patient_id_column"], config["pathology_column"])
        d = filter_resolvable_rows(d, root, img_col, dicom_lookup)
        grandes = sum(max(Image.open(resolve_image_path(root, r[img_col], dicom_lookup, "crop")).size) > 2000
                      for _, r in d.iterrows())
        print(f"{nome}: {grandes}/{len(d)} recortes com tamanho de mamografia inteira (esperado ~0)")
        if nome == "teste":
            amostra = d.sample(min(n_amostra, len(d)), random_state=0)
            mask_maior = mesmo_shape_full = 0
            tem_full = full_col in d.columns
            for _, r in amostra.iterrows():
                c = Image.open(resolve_image_path(root, r[img_col], dicom_lookup, "crop")).size
                m = Image.open(resolve_image_path(root, r[mask_col], dicom_lookup, "mask")).size
                mask_maior += int(m[0] * m[1] > c[0] * c[1])
                if tem_full:
                    f = Image.open(resolve_image_path(root, r[full_col], dicom_lookup, "full")).size
                    mesmo_shape_full += int(m == f)
            print(f"  máscara maior que o recorte: {mask_maior}/{len(amostra)} (esperado ~todas)")
            if tem_full:
                print(f"  máscara com o MESMO tamanho da mamografia inteira: {mesmo_shape_full}/{len(amostra)}")
            else:
                print(f"  (coluna '{full_col}' não existe no dataframe; colunas: {list(d.columns)})")

In [14]:
from mlflow.tracking import MlflowClient
client = MlflowClient()
HASH_COM_BUG = "9572f3c"
n = 0
for exp in client.search_experiments():
    for r in client.search_runs([exp.experiment_id], max_results=50000):
        if r.data.params.get("code_version", "").startswith(HASH_COM_BUG) and not r.data.tags.get("invalid_reason"):
            client.set_tag(r.info.run_id, "invalid_reason", "dicom_lookup_bug_recorte_vs_mascara")
            client.set_tag(r.info.run_id, "status", "invalid")
            n += 1
print(n, "runs marcados como inválidos")

30 runs marcados como inválidos


### Métricas e reprodutibilidade (seeds, AUROC/AUPRC, bootstrap CI)

In [11]:
# utils.py
"""Utilidades de reprodutibilidade e métricas (AUROC, AUPRC, sensibilidade/
especificidade em ponto de operação fixo, com intervalos de confiança de 95%
por bootstrap), conforme exigido no protocolo experimental do projeto."""

import random

import numpy as np
import torch
from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve,
    confusion_matrix, f1_score,
)


def set_seed(seed: int):
    """Fixa a semente em todas as fontes de aleatoriedade relevantes."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Determinismo adicional (pode reduzir performance um pouco, mas é mais
    # importante para reprodutibilidade em um projeto acadêmico):
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def sensitivity_at_specificity(y_true, y_score, target_specificity=0.90):
    """Retorna (sensibilidade, limiar) no ponto em que a especificidade é
    a mais próxima possível (por cima) do valor alvo (padrão: 90%)."""
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    specificity = 1 - fpr
    valid = specificity >= target_specificity
    if not valid.any():
        # nenhum ponto atinge a especificidade alvo -> usa o mais próximo
        idx = np.argmax(specificity)
    else:
        # entre os pontos que atingem a especificidade alvo, pega o de
        # maior sensibilidade (tpr)
        candidate_idx = np.where(valid)[0]
        idx = candidate_idx[np.argmax(tpr[candidate_idx])]
    return float(tpr[idx]), float(thresholds[idx])


def specificity_at_sensitivity(y_true, y_score, target_sensitivity=0.90):
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    specificity = 1 - fpr
    valid = tpr >= target_sensitivity
    if not valid.any():
        idx = np.argmax(tpr)
    else:
        candidate_idx = np.where(valid)[0]
        idx = candidate_idx[np.argmax(specificity[candidate_idx])]
    return float(specificity[idx]), float(thresholds[idx])


def compute_metrics(y_true, y_score, threshold=0.5):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    y_pred = (y_score >= threshold).astype(int)

    auroc = roc_auc_score(y_true, y_score)
    auprc = average_precision_score(y_true, y_score)
    sens90, thr_sens90 = sensitivity_at_specificity(y_true, y_score, 0.90)
    spec90, thr_spec90 = specificity_at_sensitivity(y_true, y_score, 0.90)
    cm = confusion_matrix(y_true, y_pred).tolist()
    f1 = f1_score(y_true, y_pred)

    return {
        "auroc": auroc,
        "auprc": auprc,
        "sensitivity_at_specificity_90": sens90,
        "specificity_at_sensitivity_90": spec90,
        "f1_at_threshold_0.5": f1,
        "confusion_matrix_at_threshold_0.5": cm,
        "n_samples": len(y_true),
        "n_positive": int(y_true.sum()),
    }


def bootstrap_ci(y_true, y_score, metric_fn, n_bootstrap=1000, ci=0.95, seed=42):
    """Intervalo de confiança por bootstrap (reamostragem com reposição),
    conforme exigido no protocolo (mínimo 1000 reamostragens)."""
    rng = np.random.RandomState(seed)
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    n = len(y_true)

    values = []
    for _ in range(n_bootstrap):
        idx = rng.randint(0, n, n)
        y_true_bs = y_true[idx]
        y_score_bs = y_score[idx]
        if len(np.unique(y_true_bs)) < 2:
            continue  # bootstrap sem as duas classes -> métrica indefinida, pula
        values.append(metric_fn(y_true_bs, y_score_bs))

    values = np.array(values)
    alpha = 1 - ci
    lower = float(np.percentile(values, 100 * alpha / 2))
    upper = float(np.percentile(values, 100 * (1 - alpha / 2)))
    return lower, upper, float(np.mean(values))


def full_metrics_with_ci(y_true, y_score, threshold=0.5, n_bootstrap=1000, seed=42):
    """Métricas completas + IC 95% por bootstrap para AUROC e AUPRC."""
    base = compute_metrics(y_true, y_score, threshold=threshold)

    auroc_lo, auroc_hi, auroc_mean = bootstrap_ci(
        y_true, y_score, roc_auc_score, n_bootstrap=n_bootstrap, seed=seed
    )
    auprc_lo, auprc_hi, auprc_mean = bootstrap_ci(
        y_true, y_score, average_precision_score, n_bootstrap=n_bootstrap, seed=seed
    )

    base["auroc_ci95"] = [auroc_lo, auroc_hi]
    base["auprc_ci95"] = [auprc_lo, auprc_hi]
    base["bootstrap_n"] = n_bootstrap

    return base


### Conexão ao MLflow

In [12]:
import shutil, os

os.makedirs('/kaggle/working/mlflow-server', exist_ok=True)

origem = '/kaggle/input/datasets/joovitorbach/mlflowdb/mlflow.db'
destino = '/kaggle/working/mlflow-server/mlflow.db'  # precisa bater com MLFLOW_TRACKING_URI

assert os.path.exists(origem), f"Arquivo de origem não encontrado: {origem}"
shutil.copy(origem, destino)

print(f"Histórico do MLflow restaurado de '{origem}' para '{destino}'.")
print(f"Tamanho copiado: {os.path.getsize(destino)} bytes,")

Histórico do MLflow restaurado de '/kaggle/input/datasets/joovitorbach/mlflowdb/mlflow.db' para '/kaggle/working/mlflow-server/mlflow.db'.
Tamanho copiado: 1863680 bytes,


In [13]:
# ============================================================
# Conexão ao MLflow — segue o mesmo padrão do restante do grupo
# (scripts/07_train_baseline.py, scripts/08_mlflow_smoke_test.py):
# grava direto em um banco SQLite local, SEM precisar de um servidor
# remoto compartilhado rodando. Isso funciona igual dentro do Kaggle.
#
# Se, no futuro, o grupo decidir usar um servidor MLflow remoto de
# verdade (não localhost — teria que ser uma URL pública), basta
# cadastrar o Secret MLFLOW_TRACKING_URI que este notebook já usa ele
# automaticamente no lugar do SQLite local.
# ============================================================
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mlflow"], check=True)

import mlflow

MLFLOW_TRACKING_URI = None
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    MLFLOW_TRACKING_URI = secrets.get_secret("MLFLOW_TRACKING_URI")
    if secrets.get_secret("MLFLOW_TRACKING_TOKEN"):
        os.environ["MLFLOW_TRACKING_TOKEN"] = secrets.get_secret("MLFLOW_TRACKING_TOKEN")
except Exception:
    MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI")

if not MLFLOW_TRACKING_URI:
    # Sem servidor remoto configurado -> grava local, dentro do próprio
    # ambiente do Kaggle. O arquivo mlflow.db fica em /kaggle/working e
    # deve ser baixado ao final da sessão (ele NÃO é salvo automaticamente
    # depois que o notebook desliga).
    os.makedirs("/kaggle/working/mlflow-server", exist_ok=True)
    MLFLOW_TRACKING_URI = "sqlite:////kaggle/working/mlflow-server/mlflow.db"
    print("[mlflow] Nenhum servidor remoto configurado (Secret MLFLOW_TRACKING_URI ausente).")
    print(f"[mlflow] Gravando localmente em: {MLFLOW_TRACKING_URI}")
    print("[mlflow] IMPORTANTE: baixe a pasta /kaggle/working/mlflow-server ao final "
          "da sessão (Output do notebook), ou os runs somem quando a sessão encerrar.")
else:
    print(f"[mlflow] Usando servidor remoto configurado: {MLFLOW_TRACKING_URI}")

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

# Nome do experimento segue a convenção <grupo>-<tarefa>-<eixo-de-variacao>
EXPERIMENT_NAME_SMOKE = "g1-classificacao-smoke-test"
EXPERIMENT_NAME_FULL = f"g1-classificacao-{CONFIG['ablation_axis']}"

mlflow.set_experiment(EXPERIMENT_NAME_SMOKE)
print(f"Experimento (smoke test): {EXPERIMENT_NAME_SMOKE}")
print(f"Experimento (run completo): {EXPERIMENT_NAME_FULL}")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.1/51.1 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 91.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 71.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.7/148.7 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.5/228.5 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.2/

### Modelo e lógica de execução de um run (compartilhada entre smoke test e run completo)

In [ ]:
import torch
import torch.nn as nn
import subprocess
from torch.utils.data import DataLoader
from torchvision import models

# Caminho do script de avaliação compartilhado do grupo — ajuste se o seu
# dataset Kaggle tiver um nome diferente.
EVALUATE_PATH = "/kaggle/input/datasets/joovitorbach/evaluate-py-compartilhado/evaluate_shared/evaluate_shared"


def build_model(architecture, pretrained, num_classes=1):
    if architecture != "resnet50":
        raise NotImplementedError(f"Arquitetura '{architecture}' não implementada.")
    weights = models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
    model = models.resnet50(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


def run_epoch(model, loader, criterion, optimizer, device, train: bool):
    model.train() if train else model.eval()
    total_loss, all_labels, all_scores = 0.0, [], []
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for images, labels, _pid, _path in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True).unsqueeze(1)
            if train:
                optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * images.size(0)
            all_scores.extend(torch.sigmoid(logits).detach().cpu().numpy().flatten().tolist())
            all_labels.extend(labels.detach().cpu().numpy().flatten().tolist())
    metrics = compute_metrics(all_labels, all_scores)
    metrics["loss"] = total_loss / len(loader.dataset)
    return metrics


def run_shared_evaluate(predictions_df: "pd.DataFrame", run_id: str) -> dict:
    """Chama o evaluate.py compartilhado do grupo em vez de calcular as
    métricas com lógica própria — garante que a avaliação usa exatamente
    o mesmo protocolo que os outros grupos/sprints, sem escolher limiar
    no teste (isso é responsabilidade do evaluate.py, não daqui)."""
    eval_input_df = predictions_df.rename(
        columns={"patient_id": "case_id", "true_label": "y_true", "predicted_probability": "y_score"}
    )[["case_id", "y_true", "y_score"]]

    preds_path = os.path.join(WORKDIR, f"{run_id}_preds_for_eval.csv")
    metrics_out_path = os.path.join(WORKDIR, f"{run_id}_eval_metrics.json")
    eval_input_df.to_csv(preds_path, index=False)

    cmd = ["python", f"{EVALUATE_PATH}/evaluate.py",
           "--task", "classification", "--input", preds_path, "--output", metrics_out_path]
    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0 or not os.path.exists(metrics_out_path):
        print("[evaluate.py] FALHOU — stdout/stderr abaixo. Usando métricas internas como fallback.")
        print(result.stdout)
        print(result.stderr)
        os.remove(preds_path)
        return None

    with open(metrics_out_path, "r", encoding="utf-8") as f:
        metrics = json.load(f)

    # Loga o CSV de entrada e o metrics.json de saída como artefatos, para
    # rastreabilidade completa de qual avaliação gerou qual número.
    mlflow.log_artifact(preds_path)
    mlflow.log_artifact(metrics_out_path)
    os.remove(preds_path)
    os.remove(metrics_out_path)

    return metrics


def log_metrics_dict_to_mlflow(metrics: dict, prefix: str = ""):
    """Loga qualquer dict de métricas no MLflow, achatando valores aninhados
    (ex: listas de intervalo de confiança viram _low/_high). Genérico de
    propósito, pois não sabemos de antemão todas as chaves exatas que o
    evaluate.py compartilhado retorna."""
    for key, value in metrics.items():
        full_key = f"{prefix}{key}"
        if isinstance(value, bool):
            continue
        elif isinstance(value, (int, float)):
            mlflow.log_metric(full_key, value)
        elif isinstance(value, (list, tuple)) and len(value) == 2 and all(isinstance(v, (int, float)) for v in value):
            mlflow.log_metric(f"{full_key}_low", value[0])
            mlflow.log_metric(f"{full_key}_high", value[1])
        elif isinstance(value, dict):
            log_metrics_dict_to_mlflow(value, prefix=f"{full_key}_")
        # strings, listas de tamanho != 2, matrizes de confusão completas etc.
        # ficam só no metrics.json (artefato), não como mlflow.log_metric.


def execute_run(config: dict, smoke_test: bool, mlflow_experiment: str,
                 max_smoke_samples: int = 16, extra_tags: dict = None):
    set_seed(config["seed"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_cuda = device.type == "cuda"

    mlflow.set_experiment(mlflow_experiment)

    with mlflow.start_run(run_name=f"{config['experiment_id']}{'-smoke' if smoke_test else ''}") as run:
        mlflow.set_tags({
            "owner": config["owner"],
            "role": config["role"],
            "status": "exploratory" if smoke_test else "success",  # sobrescrito no fim/erro
            "ablation_axis": "none" if smoke_test else config["ablation_axis"],
            "environment": ENVIRONMENT,
            **(extra_tags or {}),
        })

        mlflow.log_params({
            "code_version": CODE_VERSION,
            "seed": config["seed"],
            "dataset_manifest": ",".join(config["train_csvs"] + config["test_csvs"]),
            "split_strategy": "CBIS-DDSM_official_train_test + patient-level_val_split",
            "architecture": config["architecture"],
            "pretrained_weights": config["pretrained_weights"],
            "image_size": config["image_size"],
            "batch_size": config["batch_size"],
            "learning_rate": config["learning_rate"],
            "weight_decay": config["weight_decay"],
            "epochs": 1 if smoke_test else config["epochs"],
            "preprocessing_version": config["preprocessing_version"],
            "environment": ENVIRONMENT,
        })

        try:
            print("Construindo mapa de tradução dicom_info.csv -> jpeg real...")
            dicom_lookup = build_dicom_info_lookup(config["dataset_root"])
            print(f"Mapa construído: {len(dicom_lookup)} imagens mapeadas.")

            train_full_df = build_labeled_dataframe(
                config["dataset_root"], config["train_csvs"], config["malignant_labels"],
                config["non_malignant_labels"], config["patient_id_column"], config["pathology_column"],
            )
            train_full_df = filter_resolvable_rows(train_full_df, config["dataset_root"],
                                                     config["image_path_column"], dicom_lookup)
            train_df, val_df = patient_level_split(train_full_df, config["val_fraction"], config["seed"])

            if smoke_test:
                train_df = train_df.head(max_smoke_samples)
                val_df = val_df.head(max(4, max_smoke_samples // 4))

            mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
            train_ds = CBISDDSMPatchDataset(train_df, config["dataset_root"], config["image_path_column"],
                                             config["image_size"], mean, std, augment=not smoke_test,
                                             dicom_lookup=dicom_lookup)
            val_ds = CBISDDSMPatchDataset(val_df, config["dataset_root"], config["image_path_column"],
                                           config["image_size"], mean, std, augment=False,
                                           dicom_lookup=dicom_lookup)

            # num_workers>0 paraleliza a leitura/decodificação de imagem, que é
            # o gargalo real (não a GPU) — sem isso a GPU fica ociosa esperando
            # dado. pin_memory acelera a transferência CPU->GPU quando há CUDA.
            NUM_WORKERS = 2
            train_loader = DataLoader(train_ds, batch_size=min(config["batch_size"], len(train_ds)),
                                       shuffle=True, num_workers=NUM_WORKERS, pin_memory=use_cuda,
                                       persistent_workers=NUM_WORKERS > 0)
            val_loader = DataLoader(val_ds, batch_size=min(config["batch_size"], len(val_ds)),
                                     shuffle=False, num_workers=NUM_WORKERS, pin_memory=use_cuda,
                                     persistent_workers=NUM_WORKERS > 0)

            model = build_model(config["architecture"], config["pretrained_weights"] == "imagenet").to(device)
            criterion = nn.BCEWithLogitsLoss()
            optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"],
                                          weight_decay=config["weight_decay"])
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

            n_epochs = 1 if smoke_test else config["epochs"]
            history = []
            t_start = time.time()
            best_val_auroc, checkpoint_path = 0.0, os.path.join(WORKDIR, f"{run.info.run_id}_best.pt")

            for epoch in range(1, n_epochs + 1):
                t0 = time.time()
                if use_cuda:
                    torch.cuda.reset_peak_memory_stats(device)

                train_m = run_epoch(model, train_loader, criterion, optimizer, device, train=True)
                val_m = run_epoch(model, val_loader, criterion, optimizer, device, train=False)
                scheduler.step(val_m["auroc"])
                dt = time.time() - t0

                peak_vram_mb = torch.cuda.max_memory_allocated(device) / (1024 ** 2) if use_cuda else None

                mlflow.log_metric("train_loss", train_m["loss"], step=epoch)
                mlflow.log_metric("val_loss", val_m["loss"], step=epoch)
                mlflow.log_metric("train_auroc", train_m["auroc"], step=epoch)
                mlflow.log_metric("val_auroc", val_m["auroc"], step=epoch)
                if peak_vram_mb is not None:
                    mlflow.log_metric("peak_vram_mb_epoch", peak_vram_mb, step=epoch)

                history.append({"epoch": epoch, "train_loss": train_m["loss"], "val_loss": val_m["loss"],
                                 "train_auroc": train_m["auroc"], "val_auroc": val_m["auroc"],
                                 "seconds": dt, "peak_vram_mb": peak_vram_mb})

                vram_str = f"{peak_vram_mb:.0f}MB" if peak_vram_mb is not None else "N/A (CPU)"
                print(f"Época {epoch}/{n_epochs} | val_auroc={val_m['auroc']:.4f} | {dt:.1f}s | VRAM pico: {vram_str}")

                if val_m["auroc"] > best_val_auroc:
                    best_val_auroc = val_m["auroc"]
                    torch.save({"model_state_dict": model.state_dict(), "architecture": config["architecture"],
                                "image_size": config["image_size"], "seed": config["seed"],
                                "experiment_id": config["experiment_id"]}, checkpoint_path)

            duration_seconds = sum(h["seconds"] for h in history)
            avg_epoch_seconds = duration_seconds / len(history) if history else None
            vram_values = [h["peak_vram_mb"] for h in history if h["peak_vram_mb"] is not None]

            mlflow.log_metric("duration_seconds", duration_seconds)
            if avg_epoch_seconds is not None:
                mlflow.log_metric("avg_epoch_time_seconds", avg_epoch_seconds)
            if vram_values:
                mlflow.log_metric("peak_vram_mb_max", max(vram_values))
            if use_cuda:
                mlflow.log_param("gpu_name", torch.cuda.get_device_name(device))

            history_path = os.path.join(WORKDIR, f"{run.info.run_id}_history.csv")
            pd.DataFrame(history).to_csv(history_path, index=False)
            mlflow.log_artifact(history_path)
            os.remove(history_path)

            config_path = os.path.join(WORKDIR, f"{run.info.run_id}_config.json")
            with open(config_path, "w", encoding="utf-8") as f:
                json.dump(config, f, indent=2, ensure_ascii=False)
            mlflow.log_artifact(config_path)
            os.remove(config_path)

            if not smoke_test:
                test_df = build_labeled_dataframe(
                    config["dataset_root"], config["test_csvs"], config["malignant_labels"],
                    config["non_malignant_labels"], config["patient_id_column"], config["pathology_column"],
                )
                test_df = filter_resolvable_rows(test_df, config["dataset_root"],
                                                  config["image_path_column"], dicom_lookup)
                test_ds = CBISDDSMPatchDataset(test_df, config["dataset_root"], config["image_path_column"],
                                                config["image_size"], mean, std, augment=False,
                                                dicom_lookup=dicom_lookup)
                test_loader = DataLoader(test_ds, batch_size=config["batch_size"], shuffle=False,
                                          num_workers=NUM_WORKERS, pin_memory=use_cuda)

                model.eval()
                records = []
                with torch.no_grad():
                    for images, labels, patient_ids, _paths in test_loader:
                        probs = torch.sigmoid(model(images.to(device))).cpu().numpy().flatten()
                        for pid, lab, prob in zip(patient_ids, labels.numpy().flatten(), probs):
                            records.append({"patient_id": pid, "true_label": int(lab),
                                             "predicted_probability": float(prob)})

                predictions_df = pd.DataFrame(records)

                # ---- Avaliação via evaluate.py compartilhado (protocolo único do grupo) ----
                print("Rodando evaluate.py compartilhado...")
                shared_metrics = run_shared_evaluate(predictions_df, run.info.run_id)

                if shared_metrics is not None:
                    log_metrics_dict_to_mlflow(shared_metrics)
                    mlflow.set_tag("evaluation_source", "evaluate_shared")
                    print("Métricas do evaluate.py compartilhado registradas no MLflow.")
                else:
                    # Fallback: métricas internas, só se o script compartilhado falhar
                    metrics_test = full_metrics_with_ci(
                        predictions_df["true_label"].values, predictions_df["predicted_probability"].values,
                        n_bootstrap=1000, seed=config["seed"],
                    )
                    log_metrics_dict_to_mlflow(metrics_test)
                    mlflow.set_tag("evaluation_source", "internal_fallback")
                    print("[AVISO] evaluate.py compartilhado falhou — usadas métricas internas como fallback.")

                if os.path.exists(checkpoint_path):
                    mlflow.log_artifact(checkpoint_path)
                    os.remove(checkpoint_path)

            else:
                if os.path.exists(checkpoint_path):
                    os.remove(checkpoint_path)

            mlflow.set_tag("status", "success")
            return run.info.run_id

        except Exception as e:
            # Runs falhos também ficam registrados (não são descartados nem
            # escondidos), conforme critério de aceite da ablação.
            mlflow.set_tag("status", "failed")
            mlflow.log_param("error_message", str(e)[:250])
            print(f"[ERRO] Run falhou, mas foi registrado no MLflow com status=failed: {e}")
            raise


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models


def build_model(architecture, pretrained, num_classes=1):
    if architecture != "resnet50":
        raise NotImplementedError(f"Arquitetura '{architecture}' não implementada.")
    weights = models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
    model = models.resnet50(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


def run_epoch(model, loader, criterion, optimizer, device, train: bool):
    model.train() if train else model.eval()
    total_loss, all_labels, all_scores = 0.0, [], []
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for images, labels, _pid, _path in loader:
            images, labels = images.to(device), labels.to(device).unsqueeze(1)
            if train:
                optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * images.size(0)
            all_scores.extend(torch.sigmoid(logits).detach().cpu().numpy().flatten().tolist())
            all_labels.extend(labels.detach().cpu().numpy().flatten().tolist())
    metrics = compute_metrics(all_labels, all_scores)
    metrics["loss"] = total_loss / len(loader.dataset)
    return metrics


def execute_run(config: dict, smoke_test: bool, mlflow_experiment: str,
                 max_smoke_samples: int = 16, extra_tags: dict = None):
    set_seed(config["seed"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_cuda = device.type == "cuda"  # NOVO

    mlflow.set_experiment(mlflow_experiment)

    with mlflow.start_run(run_name=f"{config['experiment_id']}{'-smoke' if smoke_test else ''}") as run:
        mlflow.set_tags({
            "owner": config["owner"],
            "role": config["role"],
            "status": "exploratory" if smoke_test else "success",
            "ablation_axis": "none" if smoke_test else config["ablation_axis"],
            "environment": ENVIRONMENT,
            **(extra_tags or {}),
        })

        mlflow.log_params({
            "code_version": CODE_VERSION,
            "seed": config["seed"],
            "dataset_manifest": ",".join(config["train_csvs"] + config["test_csvs"]),
            "split_strategy": "CBIS-DDSM_official_train_test + patient-level_val_split",
            "architecture": config["architecture"],
            "pretrained_weights": config["pretrained_weights"],
            "image_size": config["image_size"],
            "batch_size": config["batch_size"],
            "learning_rate": config["learning_rate"],
            "weight_decay": config["weight_decay"],
            "epochs": 1 if smoke_test else config["epochs"],
            "preprocessing_version": config["preprocessing_version"],
            "environment": ENVIRONMENT,
        })

        try:
            print("Construindo mapa de tradução dicom_info.csv -> jpeg real...")
            dicom_lookup = build_dicom_info_lookup(config["dataset_root"])
            print(f"Mapa construído: {len(dicom_lookup)} imagens mapeadas.")

            train_full_df = build_labeled_dataframe(
                config["dataset_root"], config["train_csvs"], config["malignant_labels"],
                config["non_malignant_labels"], config["patient_id_column"], config["pathology_column"],
            )
            train_full_df = filter_resolvable_rows(train_full_df, config["dataset_root"],
                                                     config["image_path_column"], dicom_lookup)
            train_df, val_df = patient_level_split(train_full_df, config["val_fraction"], config["seed"])

            if smoke_test:
                train_df = train_df.head(max_smoke_samples)
                val_df = val_df.head(max(4, max_smoke_samples // 4))

            mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
            train_ds = CBISDDSMPatchDataset(train_df, config["dataset_root"], config["image_path_column"],
                                             config["image_size"], mean, std, augment=not smoke_test,
                                             dicom_lookup=dicom_lookup)
            val_ds = CBISDDSMPatchDataset(val_df, config["dataset_root"], config["image_path_column"],
                                           config["image_size"], mean, std, augment=False,
                                           dicom_lookup=dicom_lookup)

            train_loader = DataLoader(train_ds, batch_size=min(config["batch_size"], len(train_ds)),
                                       shuffle=True, num_workers=2, pin_memory=use_cuda)  # NUM_WORKERS corrigido
            val_loader = DataLoader(val_ds, batch_size=min(config["batch_size"], len(val_ds)),
                                     shuffle=False, num_workers=2, pin_memory=use_cuda)

            model = build_model(config["architecture"], config["pretrained_weights"] == "imagenet").to(device)
            criterion = nn.BCEWithLogitsLoss()
            optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"],
                                          weight_decay=config["weight_decay"])
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

            n_epochs = 1 if smoke_test else config["epochs"]
            history = []
            t_start = time.time()
            best_val_auroc, checkpoint_path = 0.0, os.path.join(WORKDIR, f"{run.info.run_id}_best.pt")

            for epoch in range(1, n_epochs + 1):
                t0 = time.time()
                if use_cuda:  # NOVO: reseta o contador de pico de VRAM a cada época
                    torch.cuda.reset_peak_memory_stats(device)

                train_m = run_epoch(model, train_loader, criterion, optimizer, device, train=True)
                val_m = run_epoch(model, val_loader, criterion, optimizer, device, train=False)
                scheduler.step(val_m["auroc"])
                dt = time.time() - t0

                # NOVO: captura o pico de VRAM da época
                peak_vram_mb = torch.cuda.max_memory_allocated(device) / (1024 ** 2) if use_cuda else None

                mlflow.log_metric("train_loss", train_m["loss"], step=epoch)
                mlflow.log_metric("val_loss", val_m["loss"], step=epoch)
                mlflow.log_metric("train_auroc", train_m["auroc"], step=epoch)
                mlflow.log_metric("val_auroc", val_m["auroc"], step=epoch)
                if peak_vram_mb is not None:  # NOVO
                    mlflow.log_metric("peak_vram_mb_epoch", peak_vram_mb, step=epoch)

                history.append({"epoch": epoch, "train_loss": train_m["loss"], "val_loss": val_m["loss"],
                                 "train_auroc": train_m["auroc"], "val_auroc": val_m["auroc"],
                                 "seconds": dt, "peak_vram_mb": peak_vram_mb})  # NOVO: peak_vram_mb

                vram_str = f"{peak_vram_mb:.0f}MB" if peak_vram_mb is not None else "N/A (CPU)"
                print(f"Época {epoch}/{n_epochs} | val_auroc={val_m['auroc']:.4f} | {dt:.1f}s | VRAM pico: {vram_str}")

                if val_m["auroc"] > best_val_auroc:
                    best_val_auroc = val_m["auroc"]
                    torch.save({"model_state_dict": model.state_dict(), "architecture": config["architecture"],
                                "image_size": config["image_size"], "seed": config["seed"],
                                "experiment_id": config["experiment_id"]}, checkpoint_path)

            duration_seconds = sum(h["seconds"] for h in history)
            avg_epoch_time_seconds = duration_seconds / len(history) if history else None  # NOVO
            vram_values = [h["peak_vram_mb"] for h in history if h["peak_vram_mb"] is not None]  # NOVO

            mlflow.log_metric("duration_seconds", duration_seconds)
            if avg_epoch_time_seconds is not None:  # NOVO
                mlflow.log_metric("avg_epoch_time_seconds", avg_epoch_time_seconds)
            if vram_values:  # NOVO
                mlflow.log_metric("peak_vram_mb_max", max(vram_values))
            if use_cuda:  # NOVO
                mlflow.log_param("gpu_name", torch.cuda.get_device_name(device))

            history_path = os.path.join(WORKDIR, f"{run.info.run_id}_history.csv")
            pd.DataFrame(history).to_csv(history_path, index=False)
            mlflow.log_artifact(history_path)
            os.remove(history_path)

            config_path = os.path.join(WORKDIR, f"{run.info.run_id}_config.json")
            with open(config_path, "w", encoding="utf-8") as f:
                json.dump(config, f, indent=2, ensure_ascii=False)
            mlflow.log_artifact(config_path)
            os.remove(config_path)

            if not smoke_test:
                test_df = build_labeled_dataframe(
                    config["dataset_root"], config["test_csvs"], config["malignant_labels"],
                    config["non_malignant_labels"], config["patient_id_column"], config["pathology_column"],
                )
                test_df = filter_resolvable_rows(test_df, config["dataset_root"],
                                                  config["image_path_column"], dicom_lookup)
                test_ds = CBISDDSMPatchDataset(test_df, config["dataset_root"], config["image_path_column"],
                                                config["image_size"], mean, std, augment=False,
                                                dicom_lookup=dicom_lookup)
                test_loader = DataLoader(test_ds, batch_size=config["batch_size"], shuffle=False,
                                          num_workers=2, pin_memory=use_cuda)

                model.eval()
                records = []
                with torch.no_grad():
                    for images, labels, patient_ids, _paths in test_loader:
                        probs = torch.sigmoid(model(images.to(device))).cpu().numpy().flatten()
                        for pid, lab, prob in zip(patient_ids, labels.numpy().flatten(), probs):
                            records.append({"patient_id": pid, "true_label": int(lab),
                                             "predicted_probability": float(prob)})

                predictions_df = pd.DataFrame(records)
                metrics_test = full_metrics_with_ci(
                    predictions_df["true_label"].values, predictions_df["predicted_probability"].values,
                    n_bootstrap=1000, seed=config["seed"],
                )

                mlflow.log_metrics({
                    "auroc": metrics_test["auroc"], "auprc": metrics_test["auprc"],
                    "sensitivity_at_90_specificity": metrics_test["sensitivity_at_specificity_90"],
                    "specificity_at_90_sensitivity": metrics_test["specificity_at_sensitivity_90"],
                    "f1": metrics_test["f1_at_threshold_0.5"],
                    "bootstrap_auroc_lower": metrics_test["auroc_ci95"][0],
                    "bootstrap_auroc_upper": metrics_test["auroc_ci95"][1],
                    "bootstrap_auprc_lower": metrics_test["auprc_ci95"][0],
                    "bootstrap_auprc_upper": metrics_test["auprc_ci95"][1],
                })
                tn, fp, fn, tp = 0, 0, 0, 0
                cm = metrics_test["confusion_matrix_at_threshold_0.5"]
                if len(cm) == 2:
                    tn, fp = cm[0]
                    fn, tp = cm[1]
                mlflow.log_metrics({"tp": tp, "fp": fp, "tn": tn, "fn": fn})

                metrics_path = os.path.join(WORKDIR, f"{run.info.run_id}_metrics.json")
                with open(metrics_path, "w", encoding="utf-8") as f:
                    json.dump(metrics_test, f, indent=2, ensure_ascii=False)
                mlflow.log_artifact(metrics_path)
                os.remove(metrics_path)

                preds_path = os.path.join(WORKDIR, f"{run.info.run_id}_predictions.csv")
                predictions_df.to_csv(preds_path, index=False)
                mlflow.log_artifact(preds_path)
                os.remove(preds_path)

                if os.path.exists(checkpoint_path):
                    mlflow.log_artifact(checkpoint_path)
                    os.remove(checkpoint_path)

                print(f"AUROC teste: {metrics_test['auroc']:.4f} "
                      f"(IC95%: {metrics_test['auroc_ci95'][0]:.4f}-{metrics_test['auroc_ci95'][1]:.4f})")
            else:
                if os.path.exists(checkpoint_path):
                    os.remove(checkpoint_path)

            mlflow.set_tag("status", "success")
            return run.info.run_id

        except Exception as e:
            mlflow.set_tag("status", "failed")
            mlflow.log_param("error_message", str(e)[:250])
            raise

## Smoke test (obrigatório antes do run completo)

Roda uma versão reduzida (poucas imagens, 1 época) só para validar que o
pipeline inteiro funciona neste ambiente — dataset carrega, modelo treina,
MLflow registra. Tag `status=exploratory`, fica isolado no experimento
`g1-classificacao-smoke-test`, nunca misturado com resultados de comparação
real.

In [ ]:
import torch
print("GPU disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Nome da GPU:", torch.cuda.get_device_name(0))
    print("Capacidade:", torch.cuda.get_device_capability(0))
print("=== SMOKE TEST — validando o pipeline antes do run completo ===")
smoke_run_id = execute_run(
    CONFIG, smoke_test=True, mlflow_experiment=EXPERIMENT_NAME_SMOKE,
    max_smoke_samples=16,
)
print(f"\nSmoke test concluído com sucesso. Run ID: {smoke_run_id}")
print("Pipeline validado — pode prosseguir para o run completo na próxima célula.")


## Run completo

**Só rode esta célula depois que o smoke test acima terminou sem erro.**
Treina com a configuração completa de `CONFIG`, avalia no teste oficial
congelado, e registra tudo no MLflow (experimento
`g1-classificacao-<eixo>`).

In [ ]:
# Só chegue até aqui depois que a célula do smoke test rodou sem erro.
print("=== RUN COMPLETO ===")
full_run_id = execute_run(
    CONFIG, smoke_test=False, mlflow_experiment=EXPERIMENT_NAME_FULL,
)
print(f"\nRun completo concluído. Run ID: {full_run_id}")
print(f"Confira os resultados no MLflow, experimento '{EXPERIMENT_NAME_FULL}'.")


 **UNet do zero vs UNet-ResNet34 — comparação em 3 seeds, protocolo congelado.**


>  ESTE É UM PIPELINE NOVO, não uma ablação em cima do que já existe. O projeto até agora só tinha um CLASSIFICADOR (ResNet50 -> maligno/não maligno). Aqui construímos SEGMENTAÇÃO pixel a pixel (prever a máscara da lesão), usando a máscara ROI oficial do CBIS-DDSM como rótulo. COMO USAR: 11 blocos, "### CÉLULA U<n> ###". Cole cada um em uma célula nova no Kaggle, na ordem U1 -> U11, depois de já ter rodado as células de definição do notebook original (CONFIG, set_seed, build_dicom_info_lookup, build_labeled_dataframe, filter_resolvable_rows, patient_level_split, resolve_image_path, WORKDIR, ENVIRONMENT, CODE_VERSION, client). DECISÕES DE PROTOCOLO CONGELADO que tomei (ajuste no U1 se seu grupo já tiver outra especificação definida):   - image_size = 224 (mesma resolução validada no pipeline de classificação)   - batch_size = 16 (menor que os 32 da classificação -- segmentação tem     saída densa por pixel, consome mais VRAM que um logit escalar)   - learning_rate = 1e-4, weight_decay = 1e-5, epochs = 15   - seeds = [42, 7, 123] -- as MESMAS do estudo de estabilidade EXP-05,    para ficar comparável com o resto do projeto   - loss = BCE + Dice combinada; threshold de binarização = 0.5   - "commit": o projeto não tem um hash de git disponível no Kaggle, então     uso CODE_VERSION (já usado em todo o resto do pipeline) como o proxy     de rastreabilidade de versão de código. Se seu grupo versiona em git     de verdade, troque CODE_VERSION por `!git rev-parse HEAD`.

 

In [ ]:
 ### CÉLULA V1 ###  (substitui U1)
import time
import random
import concurrent.futures as cf
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from PIL import Image
from torch.utils.data import Dataset, DataLoader
 
Image.MAX_IMAGE_PIXELS = None  # mamografias de ~13 Mpx; evita o aviso/erro de "decompression bomb"
 
client = mlflow.tracking.MlflowClient()
if "dicom_lookup" not in globals():
    dicom_lookup = build_dicom_info_lookup(CONFIG["dataset_root"])
 
TARGET_HW = (768, 512)  # (altura, largura)
 
CONFIG_SEG = {
    "owner": CONFIG["owner"], "role": CONFIG["role"],
    "dataset_root": CONFIG["dataset_root"],
    "train_csvs": CONFIG["train_csvs"], "test_csvs": CONFIG["test_csvs"],
    "patient_id_column": CONFIG["patient_id_column"],
    "full_image_column": "image file path",        # mamografia inteira
    "mask_path_column": "ROI mask file path",      # máscara ROI (mesmo tamanho da mamografia)
    "target_hw": TARGET_HW,
    "batch_size": 4,
    "learning_rate": 1e-4,
    "weight_decay": 1e-5,
    "epochs": 15,
    "val_fraction": CONFIG["val_fraction"],
    "preprocessing_version": CONFIG["preprocessing_version"],
    "code_version": CODE_VERSION,
    "threshold": 0.5,
    "use_amp": True,
}
 
SEEDS = [42, 7, 123]
ARCHITECTURES_SEG = ["unet_scratch", "unet_resnet34"]
EXPERIMENT_NAME_SEG = "g1-segmentacao-unet_vs_resnet34-v2"   # experimento NOVO, separado dos runs invalidados
 
print("=== PROTOCOLO CONGELADO (segmentação v2) ===")
for k, v in CONFIG_SEG.items():
    print(f"  {k}: {v}")
print(f"  seeds: {SEEDS} | arquiteturas: {ARCHITECTURES_SEG}")

In [ ]:
# ### CÉLULA V2 ###  (substitui U2)
# ------------------------------------------------------------------------
# Constrói (ou carrega do cache) os arrays de imagem/máscara reduzidos.
# Uma linha por MAMOGRAFIA, com a união das máscaras das lesões dela.
# ------------------------------------------------------------------------
 
def build_union_dataframe(config, csvs, dicom_lookup):
    root = config["dataset_root"]
    df = load_and_merge_csvs(root, csvs)
    full_col, mask_col, pid_col = config["full_image_column"], config["mask_path_column"], config["patient_id_column"]
    for c in (full_col, mask_col, pid_col):
        if c not in df.columns:
            raise KeyError(f"Coluna '{c}' não encontrada. Colunas disponíveis: {list(df.columns)}")
 
    rows, n_drop = [], 0
    for _, r in df.iterrows():
        fp = resolve_image_path(root, r[full_col], dicom_lookup, role="full")
        mp = resolve_image_path(root, r[mask_col], dicom_lookup, role="mask")
        if os.path.exists(fp) and os.path.exists(mp):
            rows.append({"patient_id": r[pid_col], "full_path": fp, "mask_path": mp})
        else:
            n_drop += 1
    tmp = pd.DataFrame(rows)
    grouped = (tmp.groupby("full_path", sort=True)
                  .agg(patient_id=("patient_id", "first"),
                       mask_paths=("mask_path", lambda s: sorted(set(s))))
                  .reset_index())
    print(f"[union] {len(df)} lesões (linhas do CSV) -> {len(grouped)} mamografias únicas "
          f"({n_drop} lesões descartadas por arquivo ausente).")
    return grouped
 
 
def preprocess_one(args):
    full_path, mask_paths, (H, W) = args
    img = Image.open(full_path).convert("L")
    orig_w, orig_h = img.size
    img_small = np.asarray(img.resize((W, H), Image.BILINEAR), dtype=np.uint8)
    cover = np.zeros((H, W), dtype=np.float32)
    area_full = 0
    for mp in mask_paths:
        m_bin = np.asarray(Image.open(mp).convert("L")) > 127
        area_full += int(m_bin.sum())
        m_small = Image.fromarray((m_bin * 255).astype(np.uint8)).resize((W, H), Image.BOX)
        cover = np.maximum(cover, np.asarray(m_small, dtype=np.float32) / 255.0)
    mask_small = (cover >= 0.5).astype(np.uint8)
    return img_small, mask_small, area_full, int(orig_h < orig_w)
 
 
def build_or_load_cache(tag, csvs, config, dicom_lookup, n_workers=4):
    H, W = config["target_hw"]
    cache_path = os.path.join(WORKDIR, f"seg_cache_{tag}_{H}x{W}.npz")
    if os.path.exists(cache_path):
        z = np.load(cache_path)
        print(f"[cache] {tag}: carregado de {cache_path} ({len(z['images'])} imagens).")
        return {k: z[k] for k in z.files}
 
    grouped = build_union_dataframe(config, csvs, dicom_lookup)
    jobs = [(r.full_path, r.mask_paths, (H, W)) for r in grouped.itertuples()]
    t0 = time.time()
    with cf.ThreadPoolExecutor(max_workers=n_workers) as ex:
        results = list(ex.map(preprocess_one, jobs))
    print(f"[cache] {tag}: {len(results)} imagens pré-processadas em {time.time() - t0:.0f}s.")
 
    images = np.stack([r[0] for r in results])
    masks = np.stack([r[1] for r in results])
    areas = np.array([r[2] for r in results], dtype=np.int64)
    landscape = np.array([r[3] for r in results], dtype=np.int8)
 
    vazias = masks.reshape(len(masks), -1).sum(axis=1) == 0
    keep = ~vazias
    print(f"[cache] {tag}: {int(vazias.sum())} imagens ficaram com máscara VAZIA após a redução "
          f"(lesões pequenas demais para {H}x{W}) e foram removidas; {int(keep.sum())} mantidas.")
    if landscape.sum() > 0:
        print(f"[cache] AVISO {tag}: {int(landscape.sum())} imagens em paisagem (largura > altura) "
              f"serão deformadas pelo redimensionamento para {H}x{W}.")
 
    data = {
        "images": images[keep], "masks": masks[keep], "areas": areas[keep],
        # np.array(..., dtype=str) gera strings do NumPy (sem pickle); arrays "object" quebram no np.load
        "patient_ids": np.array(grouped["patient_id"].astype(str).tolist(), dtype=str)[keep],
        "paths": np.array(grouped["full_path"].astype(str).tolist(), dtype=str)[keep],
        "n_empty_dropped": np.array([int(vazias.sum())]),
    }
    np.savez(cache_path, **data)
    print(f"[cache] {tag}: salvo em {cache_path}")
    return data
 
 
class SegArrayDataset(Dataset):
    def __init__(self, data, indices, augment):
        self.images, self.masks, self.indices, self.augment = data["images"], data["masks"], np.asarray(indices), augment
        self.mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        self.std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
 
    def __len__(self):
        return len(self.indices)
 
    def __getitem__(self, i):
        j = int(self.indices[i])
        img, msk = self.images[j], self.masks[j]
        if self.augment and torch.rand(1).item() < 0.5:   # torch.rand é semeado por worker
            img, msk = img[:, ::-1], msk[:, ::-1]
        x = torch.from_numpy(np.ascontiguousarray(img)).float().div_(255.0).unsqueeze(0).repeat(3, 1, 1)
        x = (x - self.mean) / self.std
        y = torch.from_numpy(np.ascontiguousarray(msk)).float().unsqueeze(0)
        return x, y, j
 
 
SEG_TRAIN = build_or_load_cache("train", CONFIG_SEG["train_csvs"], CONFIG_SEG, dicom_lookup)
SEG_TEST = build_or_load_cache("test", CONFIG_SEG["test_csvs"], CONFIG_SEG, dicom_lookup)
 
# ---- Sanidade: guarda contra uma regressão do bug (imagem == máscara) ----
for tag, d in [("treino", SEG_TRAIN), ("teste", SEG_TEST)]:
    frac = d["masks"].reshape(len(d["masks"]), -1).mean(axis=1)
    print(f"[sanidade] {tag}: {len(frac)} imagens | fração de pixels de lesão: "
          f"mediana={np.median(frac) * 100:.2f}% média={frac.mean() * 100:.2f}% máx={frac.max() * 100:.1f}%")
    assert frac.mean() < 0.20, ("Fração de lesão média alta demais -- provável bug de pareamento "
                                 "imagem/máscara. NÃO treine.")
    a = d["areas"]
    print(f"[sanidade] {tag}: área da lesão (px, resolução original) "
          f"p10={np.percentile(a, 10):.0f} mediana={np.median(a):.0f} p90={np.percentile(a, 90):.0f}")
 
# ---- Checagem visual: imagem com o contorno da máscara por cima ----
import matplotlib.pyplot as plt
rng_v = np.random.RandomState(0)
sel = rng_v.choice(len(SEG_TEST["images"]), size=4, replace=False)
fig, axs = plt.subplots(2, 4, figsize=(14, 10))
for k, j in enumerate(sel):
    axs[0, k].imshow(SEG_TEST["images"][j], cmap="gray"); axs[0, k].set_title("mamografia (reduzida)"); axs[0, k].axis("off")
    axs[1, k].imshow(SEG_TEST["images"][j], cmap="gray")
    axs[1, k].imshow(np.ma.masked_where(SEG_TEST["masks"][j] == 0, SEG_TEST["masks"][j]), cmap="autumn", alpha=0.6)
    axs[1, k].set_title(f"com máscara (área orig={SEG_TEST['areas'][j]} px)"); axs[1, k].axis("off")
plt.tight_layout(); plt.show()
print("CONFIRA O GRÁFICO: a mancha colorida deve cair sobre uma lesão/região da mama, "
      "e ser pequena em relação à imagem. Se cobrir quase tudo, ou cair fora da mama, NÃO treine.")

In [ ]:
 ### CÉLULA U3 ###
# ------------------------------------------------------------------------
# Loss (BCE + Dice combinada) e métricas. Dice/IoU são médias POR IMAGEM
# (padrão em segmentação médica). Sensibilidade e precisão POR PIXEL são
# acumuladas globalmente (soma de TP/FP/FN em todo o conjunto de teste,
# "micro-average") -- também convenção padrão da literatura, e documentado
# aqui pra não virar ambiguidade no relatório.
# ------------------------------------------------------------------------
 
def dice_loss(logits, targets, smooth=1.0):
    probs = torch.sigmoid(logits)
    probs_f = probs.view(probs.size(0), -1)
    targets_f = targets.view(targets.size(0), -1)
    intersection = (probs_f * targets_f).sum(dim=1)
    dice = (2 * intersection + smooth) / (probs_f.sum(dim=1) + targets_f.sum(dim=1) + smooth)
    return 1 - dice.mean()
 
 
def bce_dice_loss(logits, targets):
    bce = F.binary_cross_entropy_with_logits(logits, targets)
    return bce + dice_loss(logits, targets)
 
 
@torch.no_grad()
def batch_dice_iou(logits, targets, threshold=0.5, eps=1e-7):
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()
    preds_f = preds.view(preds.size(0), -1)
    targets_f = targets.view(targets.size(0), -1)
    tp = (preds_f * targets_f).sum(dim=1)
    fp = (preds_f * (1 - targets_f)).sum(dim=1)
    fn = ((1 - preds_f) * targets_f).sum(dim=1)
    dice = (2 * tp + eps) / (2 * tp + fp + fn + eps)
    iou = (tp + eps) / (tp + fp + fn + eps)
    return dice, iou, tp, fp, fn

In [ ]:
# ### CÉLULA U4 ###
# ------------------------------------------------------------------------
# Arquiteturas. unet_scratch: U-Net clássica, pesos aleatórios, do zero.
# unet_resnet34: encoder ResNet34 pré-treinado (ImageNet) + decoder com
# skip connections -- mesmo espírito de um Unet do pacote smp, implementado
# aqui direto com torchvision pra não adicionar dependência nova no Kaggle.
# ------------------------------------------------------------------------
 
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
 
    def forward(self, x):
        return self.block(x)
 
 
class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.conv = DoubleConv(in_ch + skip_ch, out_ch)
 
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))
 
 
class UNetScratch(nn.Module):
    """U-Net clássica (Ronneberger et al.), pesos aleatórios, do zero."""
    def __init__(self, in_channels=3, base=32):
        super().__init__()
        self.enc1 = DoubleConv(in_channels, base)
        self.enc2 = DoubleConv(base, base * 2)
        self.enc3 = DoubleConv(base * 2, base * 4)
        self.enc4 = DoubleConv(base * 4, base * 8)
        self.bottleneck = DoubleConv(base * 8, base * 16)
        self.pool = nn.MaxPool2d(2)
        self.up4 = UpBlock(base * 16, base * 8, base * 8)
        self.up3 = UpBlock(base * 8, base * 4, base * 4)
        self.up2 = UpBlock(base * 4, base * 2, base * 2)
        self.up1 = UpBlock(base * 2, base, base)
        self.out_conv = nn.Conv2d(base, 1, kernel_size=1)
 
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))
        d4 = self.up4(b, e4)
        d3 = self.up3(d4, e3)
        d2 = self.up2(d3, e2)
        d1 = self.up1(d2, e1)
        return self.out_conv(d1)
 
 
class UNetResNet34(nn.Module):
    """Encoder ResNet34 pré-treinado (ImageNet) + decoder com skip
    connections -- UNet-ResNet34."""
    def __init__(self, pretrained=True):
        super().__init__()
        weights = torchvision.models.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = torchvision.models.resnet34(weights=weights)
        self.stem = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu)  # /2, 64ch
        self.pool0 = backbone.maxpool                                           # /4
        self.layer1 = backbone.layer1   # /4,  64ch
        self.layer2 = backbone.layer2   # /8,  128ch
        self.layer3 = backbone.layer3   # /16, 256ch
        self.layer4 = backbone.layer4   # /32, 512ch
 
        self.up4 = UpBlock(512, 256, 256)
        self.up3 = UpBlock(256, 128, 128)
        self.up2 = UpBlock(128, 64, 64)
        self.up1 = UpBlock(64, 64, 32)
        self.up0 = nn.Sequential(nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
                                  nn.Conv2d(32, 16, 3, padding=1), nn.ReLU(inplace=True))
        self.out_conv = nn.Conv2d(16, 1, kernel_size=1)
 
    def forward(self, x):
        s0 = self.stem(x)
        s1 = self.layer1(self.pool0(s0))
        s2 = self.layer2(s1)
        s3 = self.layer3(s2)
        s4 = self.layer4(s3)
        d4 = self.up4(s4, s3)
        d3 = self.up3(d4, s2)
        d2 = self.up2(d3, s1)
        d1 = self.up1(d2, s0)
        d0 = self.up0(d1)
        return self.out_conv(d0)
 
 
def build_seg_model(architecture):
    if architecture == "unet_scratch":
        return UNetScratch()
    if architecture == "unet_resnet34":
        return UNetResNet34(pretrained=True)
    raise NotImplementedError(f"Arquitetura de segmentação '{architecture}' não implementada.")

In [ ]:
# ### CÉLULA V5 ###  (substitui U5)
# ------------------------------------------------------------------------
# Treino/validação/teste. Mantém U3 (bce_dice_loss, batch_dice_iou) e U4
# (build_seg_model). Loss calculada em fp32; forward em fp16 (AMP).
# ------------------------------------------------------------------------
 
def make_scaler(enabled):
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)
 
 
def run_epoch_seg(model, loader, optimizer, device, train, threshold, scaler, use_amp):
    model.train() if train else model.eval()
    total_loss, n_total, dices, ious = 0.0, 0, [], []
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for images, masks, _idx in loader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            if train:
                optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                logits = model(images)
            logits = logits.float()
            loss = bce_dice_loss(logits, masks)
            if train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            dice, iou, _, _, _ = batch_dice_iou(logits.detach(), masks, threshold)
            total_loss += loss.item() * images.size(0)
            n_total += images.size(0)
            dices.extend(dice.cpu().numpy().tolist())
            ious.extend(iou.cpu().numpy().tolist())
    return total_loss / n_total, float(np.mean(dices)), float(np.mean(ious))
 
 
def run_segmentation_experiment(architecture, seed, config, smoke_test, mlflow_experiment,
                                 max_smoke_samples=8, extra_tags=None):
    set_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_cuda = device.type == "cuda"
    use_amp = bool(config["use_amp"] and use_cuda)
    scaler = make_scaler(use_amp)
    mlflow.set_experiment(mlflow_experiment)
    run_name = f"EXP-SEG2-{architecture}-seed{seed}{'-smoke' if smoke_test else ''}"
    H, W = config["target_hw"]
 
    with mlflow.start_run(run_name=run_name) as run:
        mlflow.set_tags({
            "owner": config["owner"], "role": config["role"],
            "status": "exploratory" if smoke_test else "success",
            "ablation_axis": "architecture", "environment": ENVIRONMENT,
            "architecture": architecture, "seed": str(seed),
            "dataset_manifest": ",".join(config["train_csvs"] + config["test_csvs"]),
            "code_version": config["code_version"],
            "data_pairing": "full_mammogram+union_roi_mask", "dicom_lookup": "role_aware_v2",
            **(extra_tags or {}),
        })
        mlflow.log_params({
            "code_version": config["code_version"], "seed": seed, "architecture": architecture,
            "target_hw": f"{H}x{W}", "batch_size": config["batch_size"],
            "learning_rate": config["learning_rate"], "weight_decay": config["weight_decay"],
            "epochs": 1 if smoke_test else config["epochs"], "threshold": config["threshold"],
            "precision": "amp_fp16" if use_amp else "fp32",
            "preprocessing_version": config["preprocessing_version"],
            "n_images_train_total": len(SEG_TRAIN["images"]), "n_images_test": len(SEG_TEST["images"]),
            "n_empty_dropped_train": int(SEG_TRAIN["n_empty_dropped"][0]),
            "n_empty_dropped_test": int(SEG_TEST["n_empty_dropped"][0]),
        })
 
        try:
            split_df = pd.DataFrame({"patient_id": SEG_TRAIN["patient_ids"], "idx": np.arange(len(SEG_TRAIN["images"]))})
            tr_df, va_df = patient_level_split(split_df, config["val_fraction"], seed)
            tr_idx, va_idx = tr_df["idx"].values, va_df["idx"].values
            if smoke_test:
                tr_idx, va_idx = tr_idx[:max_smoke_samples], va_idx[:max(2, max_smoke_samples // 4)]
            mlflow.log_params({"n_train": len(tr_idx), "n_val": len(va_idx)})
 
            NUM_WORKERS = 2
            train_loader = DataLoader(SegArrayDataset(SEG_TRAIN, tr_idx, augment=not smoke_test),
                                       batch_size=min(config["batch_size"], len(tr_idx)), shuffle=True,
                                       num_workers=NUM_WORKERS, pin_memory=use_cuda, drop_last=len(tr_idx) > config["batch_size"])
            val_loader = DataLoader(SegArrayDataset(SEG_TRAIN, va_idx, augment=False),
                                     batch_size=min(config["batch_size"], len(va_idx)), shuffle=False,
                                     num_workers=NUM_WORKERS, pin_memory=use_cuda)
 
            model = build_seg_model(architecture).to(device)
            optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"],
                                          weight_decay=config["weight_decay"])
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)
 
            n_epochs = 1 if smoke_test else config["epochs"]
            history, best_val_dice = [], -1.0
            checkpoint_path = os.path.join(WORKDIR, f"{run.info.run_id}_best.pt")
 
            for epoch in range(1, n_epochs + 1):
                t0 = time.time()
                if use_cuda:
                    torch.cuda.reset_peak_memory_stats(device)
                train_loss, train_dice, train_iou = run_epoch_seg(model, train_loader, optimizer, device, True,
                                                                    config["threshold"], scaler, use_amp)
                val_loss, val_dice, val_iou = run_epoch_seg(model, val_loader, optimizer, device, False,
                                                             config["threshold"], scaler, use_amp)
                scheduler.step(val_dice)
                dt = time.time() - t0
                peak_vram = torch.cuda.max_memory_allocated(device) / (1024 ** 2) if use_cuda else None
 
                mlflow.log_metrics({"train_loss": train_loss, "val_loss": val_loss,
                                     "train_dice": train_dice, "val_dice": val_dice,
                                     "train_iou": train_iou, "val_iou": val_iou}, step=epoch)
                if peak_vram is not None:
                    mlflow.log_metric("peak_vram_mb_epoch", peak_vram, step=epoch)
                history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
                                 "train_dice": train_dice, "val_dice": val_dice, "train_iou": train_iou,
                                 "val_iou": val_iou, "seconds": dt, "peak_vram_mb": peak_vram})
                vram_str = f"{peak_vram:.0f}MB" if peak_vram is not None else "N/A (CPU)"
                print(f"[{architecture} | seed={seed}] Época {epoch}/{n_epochs} | val_dice={val_dice:.4f} "
                      f"val_iou={val_iou:.4f} | {dt:.1f}s | VRAM: {vram_str}")
 
                if val_dice > best_val_dice:
                    best_val_dice = val_dice
                    torch.save({"model_state_dict": model.state_dict(), "architecture": architecture,
                                "seed": seed}, checkpoint_path)
 
            duration_seconds = sum(h["seconds"] for h in history)
            mlflow.log_metric("duration_seconds", duration_seconds)
            mlflow.log_metric("avg_epoch_time_seconds", duration_seconds / len(history))
            vram_values = [h["peak_vram_mb"] for h in history if h["peak_vram_mb"] is not None]
            if vram_values:
                mlflow.log_metric("peak_vram_mb_max", max(vram_values))
 
            history_path = os.path.join(WORKDIR, f"{run.info.run_id}_history.csv")
            pd.DataFrame(history).to_csv(history_path, index=False)
            mlflow.log_artifact(history_path)
            os.remove(history_path)
 
            if not smoke_test:
                # avalia o MELHOR checkpoint (por val_dice), não o último estado do modelo
                ckpt = torch.load(checkpoint_path, map_location=device)
                model.load_state_dict(ckpt["model_state_dict"])
                model.eval()
                te_idx = np.arange(len(SEG_TEST["images"]))
                test_loader = DataLoader(SegArrayDataset(SEG_TEST, te_idx, augment=False),
                                          batch_size=config["batch_size"], shuffle=False,
                                          num_workers=NUM_WORKERS, pin_memory=use_cuda)
                dices_by_idx, ious_by_idx = {}, {}
                tp_tot = fp_tot = fn_tot = 0.0
                with torch.no_grad():
                    for images, masks, idxs in test_loader:
                        images, masks = images.to(device), masks.to(device)
                        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                            logits = model(images)
                        dice, iou, tp, fp, fn = batch_dice_iou(logits.float(), masks, config["threshold"])
                        for k, j in enumerate(idxs.numpy().tolist()):
                            dices_by_idx[j], ious_by_idx[j] = float(dice[k]), float(iou[k])
                        tp_tot += tp.sum().item(); fp_tot += fp.sum().item(); fn_tot += fn.sum().item()
 
                order = sorted(dices_by_idx)
                dices_all = np.array([dices_by_idx[j] for j in order])
                ious_all = np.array([ious_by_idx[j] for j in order])
                areas = SEG_TEST["areas"][order]
                precision_test = tp_tot / (tp_tot + fp_tot + 1e-7)
                sensitivity_test = tp_tot / (tp_tot + fn_tot + 1e-7)
                mlflow.log_metrics({"dice_test": float(dices_all.mean()), "iou_test": float(ious_all.mean()),
                                     "precision_test": precision_test, "sensitivity_test": sensitivity_test,
                                     "dice_test_median": float(np.median(dices_all))})
 
                q1, q2 = np.quantile(areas, [1 / 3, 2 / 3])
                buckets = {"pequena": areas <= q1, "media": (areas > q1) & (areas <= q2), "grande": areas > q2}
                for name, sel in buckets.items():
                    if sel.sum() > 0:
                        mlflow.log_metric(f"dice_test_{name}", float(dices_all[sel].mean()))
                        mlflow.log_metric(f"iou_test_{name}", float(ious_all[sel].mean()))
                        mlflow.log_metric(f"n_test_{name}", int(sel.sum()))
                mlflow.log_metrics({"area_q33_px": float(q1), "area_q67_px": float(q2)})
 
                print(f"[{architecture} | seed={seed}] TESTE -> dice={dices_all.mean():.4f} iou={ious_all.mean():.4f} "
                      f"precisao={precision_test:.4f} sensibilidade={sensitivity_test:.4f}")
                mlflow.log_artifact(checkpoint_path)
            if os.path.exists(checkpoint_path):
                os.remove(checkpoint_path)
 
            mlflow.set_tag("status", "success")
            return run.info.run_id
 
        except Exception as e:
            mlflow.set_tag("status", "failed")
            mlflow.log_param("error_message", str(e)[:250])
            print(f"[ERRO][{architecture} seed={seed}] Run falhou, registrado como status=failed: {e}")
            raise

In [ ]:
# ### CÉLULA V6 ###  (substitui U6)
# ------------------------------------------------------------------------
# Probe de VRAM com batch cheio, 768x512 e AMP, para as duas arquiteturas.
# ------------------------------------------------------------------------
 
def vram_probe_seg(architecture, batch_size, hw, use_amp):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type != "cuda":
        print(f"[probe] Sem GPU -- pulando probe para {architecture}.")
        return None
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(device)
    model = build_seg_model(architecture).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    scaler = make_scaler(use_amp)
    x = torch.randn(batch_size, 3, *hw, device=device)
    y = torch.randint(0, 2, (batch_size, 1, *hw), device=device).float()
    optimizer.zero_grad()
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
        logits = model(x)
    loss = bce_dice_loss(logits.float(), y)
    scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
    peak_mb = torch.cuda.max_memory_allocated(device) / (1024 ** 2)
    print(f"[probe] {architecture} | batch={batch_size} {hw[0]}x{hw[1]} amp={use_amp} -> pico de VRAM: {peak_mb:.0f} MB")
    del model, optimizer, x, y, logits, loss
    torch.cuda.empty_cache()
    return peak_mb
 
 
print("=== PROBE DE VRAM (segmentação v2) ===")
for arch in ARCHITECTURES_SEG:
    vram_probe_seg(arch, CONFIG_SEG["batch_size"], CONFIG_SEG["target_hw"], CONFIG_SEG["use_amp"])

In [ ]:
# ### CÉLULA V7 ###  (substitui U7)
# ------------------------------------------------------------------------
# PILOTO PRIMEIRO: por padrão roda só a seed 42 das duas arquiteturas (2 runs)
# para você medir o tempo por época antes de gastar GPU com as 6 combinações.
# Depois do piloto, troque SEEDS_ATIVAS para SEEDS e rode de novo -- combos já
# concluídos com sucesso são pulados.
# ------------------------------------------------------------------------
 
SEEDS_ATIVAS = [42]          # piloto. Para o estudo completo: SEEDS_ATIVAS = SEEDS
EXPERIMENT_NAME_SMOKE = "g1-classificacao-smoke-test"
seg_run_ids = globals().get("seg_run_ids", {})
 
for arch in ARCHITECTURES_SEG:
    for seed in SEEDS_ATIVAS:
        chave = f"{arch}-seed{seed}"
        if seg_run_ids.get(chave, {}).get("status") == "success":
            print(f"[skip] {chave} já concluído."); continue
        print(f"\n=== SMOKE TEST — {chave} ===")
        try:
            sid = run_segmentation_experiment(arch, seed, CONFIG_SEG, smoke_test=True,
                                               mlflow_experiment=EXPERIMENT_NAME_SMOKE)
            print(f"Smoke OK. run_id={sid}")
        except Exception as e:
            print(f"[AVISO] Smoke de {chave} falhou: {e}")
            seg_run_ids[chave] = {"run_id": None, "architecture": arch, "seed": seed,
                                   "status": "failed_smoke", "error": str(e)[:200]}
            continue
 
        print(f"\n{'=' * 60}\nRodando {chave}\n{'=' * 60}")
        try:
            run_id = run_segmentation_experiment(arch, seed, CONFIG_SEG, smoke_test=False,
                                                  mlflow_experiment=EXPERIMENT_NAME_SEG)
            seg_run_ids[chave] = {"run_id": run_id, "architecture": arch, "seed": seed, "status": "success"}
            print(f"OK — {chave} concluído. run_id: {run_id}")
        except Exception as e:
            seg_run_ids[chave] = {"run_id": None, "architecture": arch, "seed": seed,
                                   "status": "failed", "error": str(e)[:200]}
            print(f"FALHOU — {chave}: {e}")
 
print("\n=== RESUMO ===")
for chave, info in seg_run_ids.items():
    print(f"  {chave}: {info['status']} | run_id: {info.get('run_id')}")

In [ ]:
# ### CÉLULA U8 ###
# ------------------------------------------------------------------------
# Entrega 1: tabela com Dice, IoU, sensibilidade e precisão por pixel, por
# seed e por tamanho de lesão (as colunas dice_test_pequena/media/grande já
# saem prontas do próprio run_segmentation_experiment).
# ------------------------------------------------------------------------
 
linhas_seg = []
for chave, info in seg_run_ids.items():
    row = {"combo": chave, "architecture": info.get("architecture"), "seed": info.get("seed"),
           "status": info["status"], "run_id": info.get("run_id") or "N/A"}
    if info.get("run_id"):
        run = client.get_run(info["run_id"])
        m = run.data.metrics
        row.update({
            "dice_test": m.get("dice_test"), "iou_test": m.get("iou_test"),
            "precision_test": m.get("precision_test"), "sensitivity_test": m.get("sensitivity_test"),
            "dice_test_pequena": m.get("dice_test_pequena"), "dice_test_media": m.get("dice_test_media"),
            "dice_test_grande": m.get("dice_test_grande"),
            "avg_epoch_time_seconds": m.get("avg_epoch_time_seconds"),
            "peak_vram_mb_max": m.get("peak_vram_mb_max"),
        })
    linhas_seg.append(row)
 
tabela_seg = pd.DataFrame(linhas_seg).sort_values(["architecture", "seed"])
tabela_seg_path = os.path.join(WORKDIR, "tabela_unet_vs_resnet34.csv")
tabela_seg.to_csv(tabela_seg_path, index=False)
 
print("=== TABELA POR SEED E TAMANHO DE LESÃO ===")
print(tabela_seg.to_string(index=False))
print(f"\nSalva em: {tabela_seg_path}")

In [ ]:
# ### CÉLULA U9 ###
# ------------------------------------------------------------------------
# Estabilidade entre seeds (por arquitetura) + custos agregados + falhas.
# ------------------------------------------------------------------------
 
resumo_arch = []
for arch in ARCHITECTURES_SEG:
    sub = tabela_seg[(tabela_seg["architecture"] == arch) & (tabela_seg["status"] == "success")]
    falhas_arch = [c for c, i in seg_run_ids.items() if i.get("architecture") == arch and i["status"] != "success"]
    if len(sub) == 0:
        resumo_arch.append({"architecture": arch, "n_seeds_ok": 0, "falhas": len(falhas_arch),
                             "dice_mean": np.nan, "dice_std": np.nan, "dice_cv": np.nan})
        continue
    dice_mean, dice_std = sub["dice_test"].mean(), sub["dice_test"].std()
    resumo_arch.append({
        "architecture": arch, "n_seeds_ok": len(sub), "falhas": len(falhas_arch),
        "dice_mean": dice_mean, "dice_std": dice_std,
        "dice_cv": (dice_std / dice_mean) if dice_mean else np.nan,
        "iou_mean": sub["iou_test"].mean(), "iou_std": sub["iou_test"].std(),
        "avg_epoch_time_seconds_mean": sub["avg_epoch_time_seconds"].mean(),
        "peak_vram_mb_max_mean": sub["peak_vram_mb_max"].mean(),
    })
 
resumo_arch_df = pd.DataFrame(resumo_arch)
resumo_path = os.path.join(WORKDIR, "estabilidade_e_custo_por_arquitetura.csv")
resumo_arch_df.to_csv(resumo_path, index=False)
 
print("=== ESTABILIDADE ENTRE SEEDS E CUSTO, POR ARQUITETURA ===")
print(resumo_arch_df.to_string(index=False))
print(f"\nSalva em: {resumo_path}")
 
falhas_totais = [(c, i) for c, i in seg_run_ids.items() if i["status"] != "success"]
print(f"\n=== FALHAS ({len(falhas_totais)}) ===")
for chave, info in falhas_totais:
    print(f"  {chave}: {info['status']} -- {info.get('error', 'sem detalhe')}")

In [ ]:
# ### CÉLULA U10 ###
# ------------------------------------------------------------------------
# Decisão técnica -- gerada a partir dos números agregados do U9. Isso é
# um RASCUNHO estruturado para revisão humana, não uma decisão automática
# definitiva -- leiam os números reais antes de bater o martelo.
# ------------------------------------------------------------------------
 
validos = resumo_arch_df[resumo_arch_df["n_seeds_ok"] > 0].copy()
if len(validos) == 0:
    print("Nenhuma arquitetura completou ao menos 1 seed com sucesso -- sem dados para decisão técnica.")
else:
    melhor_dice = validos.loc[validos["dice_mean"].idxmax()]
    mais_estavel = validos.loc[validos["dice_cv"].idxmin()]
    mais_barata = validos.loc[validos["avg_epoch_time_seconds_mean"].idxmin()]
 
    print("=== RASCUNHO DE DECISÃO TÉCNICA ===")
    print(f"- Melhor Dice médio: {melhor_dice['architecture']} "
          f"(dice={melhor_dice['dice_mean']:.4f} ± {melhor_dice['dice_std']:.4f}, "
          f"n_seeds={int(melhor_dice['n_seeds_ok'])})")
    print(f"- Mais estável entre seeds (menor CV de Dice): {mais_estavel['architecture']} "
          f"(CV={mais_estavel['dice_cv']:.4f})")
    print(f"- Mais barata por época: {mais_barata['architecture']} "
          f"({mais_barata['avg_epoch_time_seconds_mean']:.1f}s/época, "
          f"VRAM pico médio {mais_barata['peak_vram_mb_max_mean']:.0f}MB)")
 
    if melhor_dice["architecture"] == mais_estavel["architecture"]:
        recomendacao = (f"{melhor_dice['architecture']} é recomendada: melhor Dice médio E mais estável "
                         f"entre seeds. Verificar se o custo adicional (se houver) é aceitável para o projeto.")
    else:
        recomendacao = (f"Há um trade-off: {melhor_dice['architecture']} tem melhor Dice médio, mas "
                         f"{mais_estavel['architecture']} é mais estável entre seeds. Decisão final deve "
                         f"pesar se a diferença de Dice é clinicamente relevante frente ao ganho de "
                         f"previsibilidade/estabilidade -- revisar com o grupo antes de fechar.")
    print(f"\nRecomendação preliminar: {recomendacao}")

In [ ]:
# ### CÉLULA U11 ###
# ------------------------------------------------------------------------
# Registro final no MLflow -- rastreabilidade ao manifesto, "commit"
# (code_version) e seed de cada run já está em cada run individual (U5);
# aqui registramos a análise consolidada como evidência.
# ------------------------------------------------------------------------
 
mlflow.set_experiment(EXPERIMENT_NAME_SEG)
with mlflow.start_run(run_name="EXP-SEG-analise-final"):
    mlflow.set_tags({
        "owner": CONFIG["owner"], "role": CONFIG["role"], "status": "success",
        "run_type": "unet_vs_resnet34_analysis", "environment": ENVIRONMENT,
        "code_version": CODE_VERSION,
        "dataset_manifest": ",".join(CONFIG_SEG["train_csvs"] + CONFIG_SEG["test_csvs"]),
        "seeds": ",".join(str(s) for s in SEEDS),
    })
    mlflow.log_artifact(tabela_seg_path)
    mlflow.log_artifact(resumo_path)
 
print("\nAnálise UNet vs UNet-ResNet34 registrada no MLflow (run 'EXP-SEG-analise-final').")
print(f"Tabela por seed/tamanho: {tabela_seg_path}")
print(f"Estabilidade/custo: {resumo_path}")

In [ ]:
root = CONFIG_SEG["dataset_root"]
img_col, mask_col = CONFIG_SEG["image_path_column"], CONFIG_SEG["mask_path_column"]

diag_df = build_labeled_dataframe(root, CONFIG_SEG["test_csvs"], CONFIG_SEG["malignant_labels"],
                                   CONFIG_SEG["non_malignant_labels"], CONFIG_SEG["patient_id_column"],
                                   CONFIG_SEG["pathology_column"])
diag_df = filter_resolvable_rows(diag_df, root, img_col, dicom_lookup)
diag_df = filter_rows_with_mask(diag_df, CONFIG_SEG, dicom_lookup)
amostra = diag_df.sample(min(200, len(diag_df)), random_state=0)

fracs, uniques, same_shape, parecidas, baseline_dice = [], [], 0, 0, []
pares = []
for _, row in amostra.iterrows():
    ip = resolve_image_path(root, row[img_col], dicom_lookup)
    mp = resolve_image_path(root, row[mask_col], dicom_lookup)
    img = np.array(Image.open(ip).convert("L"))
    msk = np.array(Image.open(mp).convert("L"))
    f = float((msk > 10).mean())
    fracs.append(f)
    uniques.append(len(np.unique(msk)))
    baseline_dice.append(2 * f / (1 + f))
    if img.shape == msk.shape:
        same_shape += 1
        if np.abs(img.astype(float) - msk.astype(float)).mean() < 5:
            parecidas += 1
    if len(pares) < 4:
        pares.append((img, msk))

print(f"Amostra: {len(amostra)} linhas")
print(f"Fração média de pixels positivos na máscara: {np.mean(fracs):.3f} (mediana {np.median(fracs):.3f})")
print(f"Valores únicos por máscara (mediana): {int(np.median(uniques))}  [máscara binária real ~2]")
print(f"Imagem e máscara com mesmo shape: {same_shape}/{len(amostra)}")
print(f"Máscara praticamente IGUAL à imagem: {parecidas}/{len(amostra)}")
print(f"Dice do baseline 'prever tudo como lesão': {np.mean(baseline_dice):.4f}")

fig, axs = plt.subplots(2, 4, figsize=(14, 7))
for j, (im, mk) in enumerate(pares):
    axs[0, j].imshow(im, cmap="gray"); axs[0, j].set_title(f"imagem {im.shape}"); axs[0, j].axis("off")
    axs[1, j].imshow(mk, cmap="gray"); axs[1, j].set_title(f"máscara {mk.shape}"); axs[1, j].axis("off")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(2, 4, figsize=(14, 7))
for j, (im, mk) in enumerate(pares):
    axs[0, j].imshow(im, cmap="gray"); axs[0, j].set_title(f"imagem {im.shape}"); axs[0, j].axis("off")
    axs[1, j].imshow(mk, cmap="gray"); axs[1, j].set_title(f"máscara {mk.shape}"); axs[1, j].axis("off")
plt.show()

In [ ]:
import os
import pandas as pd
from PIL import Image

root = CONFIG["dataset_root"]
info = pd.read_csv(os.path.join(root, "csv/dicom_info.csv"))
info.columns = [c.strip() for c in info.columns]
print("Colunas:", list(info.columns))
print("Linhas:", len(info), "| SeriesInstanceUID únicos:", info["SeriesInstanceUID"].nunique())
print("\nQuantos arquivos por UID:")
print(info.groupby("SeriesInstanceUID").size().value_counts())

cols_show = [c for c in ["SeriesInstanceUID", "SeriesDescription", "image_path"] if c in info.columns]
if "SeriesDescription" in info.columns:
    print("\nSeriesDescription:")
    print(info["SeriesDescription"].value_counts())
dup = info.groupby("SeriesInstanceUID").filter(lambda g: len(g) > 1)
print("\nExemplos de UID com mais de 1 arquivo:")
print(dup[cols_show].head(6).to_string())

# caminhos brutos das duas colunas, lado a lado
img_col, mask_col = CONFIG["image_path_column"], "ROI mask file path"
df_t = build_labeled_dataframe(root, CONFIG["test_csvs"], CONFIG["malignant_labels"],
                                CONFIG["non_malignant_labels"], CONFIG["patient_id_column"],
                                CONFIG["pathology_column"])
print("\nCaminhos brutos (3 linhas):")
for _, r in df_t.head(3).iterrows():
    print("  crop:", r[img_col]); print("  mask:", r[mask_col]); print()

# contaminação do classificador: arquivo resolvido tem tamanho de mamografia inteira?
if "dicom_lookup" not in globals():
    dicom_lookup = build_dicom_info_lookup(root)
for nome, csvs in [("treino", CONFIG["train_csvs"]), ("teste", CONFIG["test_csvs"])]:
    d = build_labeled_dataframe(root, csvs, CONFIG["malignant_labels"], CONFIG["non_malignant_labels"],
                                 CONFIG["patient_id_column"], CONFIG["pathology_column"])
    d = filter_resolvable_rows(d, root, img_col, dicom_lookup)
    grandes = sum(max(Image.open(resolve_image_path(root, r[img_col], dicom_lookup)).size) > 2000
                  for _, r in d.iterrows())
    print(f"{nome}: {grandes}/{len(d)} linhas em que a 'imagem do classificador' tem tamanho de "
          f"mamografia inteira (provável máscara)")